## **Imports**

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

from collections import Counter
import warnings
warnings.filterwarnings("ignore")

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## **Configurations**

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
CLAIMS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/claims_train.jsonl"
CORPUS_PATH = "/kaggle/input/datasets/anarvaaa/original-scifact-data/corpus.jsonl"

## **Load Data**

In [5]:
claims = []

with open(CLAIMS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        claims.append(json.loads(line))

print("Number of claims:", len(claims))
print(claims[0])

Number of claims: 809
{'id': 0, 'claim': '0-dimensional biomaterials lack inductive properties.', 'evidence': {}, 'cited_doc_ids': [31715818]}


In [6]:
corpus = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

print("Number of documents:", len(corpus))
print(corpus[0])

Number of documents: 5183
{'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the

In [7]:
doc_lookup = {}

for doc in corpus:
    doc_lookup[int(doc["doc_id"])] = doc

print("Documents in lookup:", len(doc_lookup))

Documents in lookup: 5183


In [8]:
example_doc_id = list(doc_lookup.keys())[0]

print("Doc ID:", example_doc_id)
print("Title:", doc_lookup[example_doc_id]["title"])
print("Abstract:")
print(doc_lookup[example_doc_id]["abstract"][:3])

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Abstract:
['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.']


## **Load SciBERT**

In [9]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='allenai/scibert_scivocab_uncased', vocab_size=31090, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	104: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## **Get first 200 tokens Abstract**

In [10]:
def get_first_tokens(document_abstract):
    
    # Convert list of abstract sentences into one document
    document_text = " ".join(document_abstract)

    # Tokenize WITHOUT special tokens
    tokens = tokenizer.tokenize(document_text)

    # Keep first 200 SciBERT tokens
    first_tokens = tokens[:400]

    # Convert tokens back to text
    first_text = tokenizer.convert_tokens_to_string(
        first_tokens
    )

    return first_text

In [11]:
test_doc = corpus[0]

first_200 = get_first_tokens(
    test_doc["abstract"]
)

print(first_200)

Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion - weighted magnetic resonance imaging ( [UNK] ) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three - dimensional fiber architecture in cerebral white matter in preterm ( n = 17 ) and full - term infants ( n = 7 ). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants ( n = 10 ) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1. 8 microm2 / ms, and decreased toward term to 1. 2 microm2 / ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar ( 1. 2 versus 1. 1 microm2 / ms ). Relative anisotropy was higher the closer birth

In [12]:
'''
claims_with_evidence = [
    item for item in claims
]

print("Original claims:", len(claims))
print("Claims with evidence:", len(claims_with_evidence))
print("Skipped:", len(claims) - len(claims_with_evidence))
'''

'\nclaims_with_evidence = [\n    item for item in claims\n]\n\nprint("Original claims:", len(claims))\nprint("Claims with evidence:", len(claims_with_evidence))\nprint("Skipped:", len(claims) - len(claims_with_evidence))\n'

## **Label Encoding**

In [13]:
label2id = {
    "CONTRADICT": 0,
    "SUPPORT": 1,
    "NEI": 2
}

id2label = {
    0: "CONTRADICT",
    1: "SUPPORT",
    2: "NEI"
}

## **Create Training Examples**

In [14]:
examples = []
skipped_mixed = 0
missing_docs = 0

for item in claims:
    claim_id = item["id"]
    claim = item["claim"]
    evidence = item.get("evidence", {})
    cited_docs = item.get("cited_doc_ids", [])

    if evidence:
        for doc_id_str, evidence_list in evidence.items():
            doc_id = int(doc_id_str)

            if doc_id not in doc_lookup:
                missing_docs += 1
                continue

            document = doc_lookup[doc_id]
            labels = set(e["label"] for e in evidence_list)

            if labels == {"SUPPORT"}:
                doc_label = "SUPPORT"
            elif labels == {"CONTRADICT"}:
                doc_label = "CONTRADICT"
            else:
                skipped_mixed += 1
                continue

            # Uniformly use full abstract text
            text = " ".join(document["abstract"])

            examples.append({
                "claim_id": claim_id,
                "doc_id": doc_id,
                "claim": claim,
                "document_text": text,
                "label": doc_label
            })
    else:
        for doc_id in cited_docs:
            if doc_id in doc_lookup:
                document = doc_lookup[doc_id]
                text = " ".join(document["abstract"])

                examples.append({
                    "claim_id": claim_id,
                    "doc_id": doc_id,
                    "claim": claim,
                    "document_text": text,
                    "label": "NEI"
                })

df = pd.DataFrame(examples)
df["label_id"] = df["label"].map(label2id)

print("Total DataFrame samples:", len(df))
print("\nClass Counts:")
print(df["label"].value_counts())

Total DataFrame samples: 894

Class Counts:
label
SUPPORT       370
NEI           330
CONTRADICT    194
Name: count, dtype: int64


In [15]:
df = pd.DataFrame(examples)
print(df.shape)
df.head()

df["label_id"] = df["label"].map(label2id)
print("Total DataFrame samples:", len(df))
print("\nClass Counts:")
print(df["label"].value_counts())
print("\nPercentages:")
print(df["label"].value_counts(normalize=True) * 100)

(894, 5)
Total DataFrame samples: 894

Class Counts:
label
SUPPORT       370
NEI           330
CONTRADICT    194
Name: count, dtype: int64

Percentages:
label
SUPPORT       41.387025
NEI           36.912752
CONTRADICT    21.700224
Name: proportion, dtype: float64


## **Train-Val Split**

In [16]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

Training examples: 715
Validation examples: 179

Training distribution:
label
SUPPORT       296
NEI           264
CONTRADICT    155
Name: count, dtype: int64

Validation distribution:
label
SUPPORT       74
NEI           66
CONTRADICT    39
Name: count, dtype: int64


## **Dataset Class**

In [17]:
class SciFactDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Tokenize pair properly into [CLS] claim [SEP] document_text [SEP]
        encoding = self.tokenizer(
            row["claim"],
            row["document_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0) 
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            row["label_id"], 
            dtype=torch.long
        )

        return item

In [18]:
train_dataset = SciFactDataset(
    train_df,
    tokenizer,
    max_length=512
)

val_dataset = SciFactDataset(
    val_df,
    tokenizer,
    max_length=512
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 715
Validation: 179


## **Check**

In [19]:
sample = train_dataset[0]

print(sample.keys())
print("Input shape:", sample["input_ids"].shape)
print("Label:", sample["labels"].item())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
Input shape: torch.Size([512])
Label: 0


In [20]:
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )
)

[CLS] There is no increased risk of hypospadias with clomiphene. [SEP] Clomifene is widely used for inducing ovulation. 1 It is structurally related to diethylstilbestrol, which has been linked to vaginal and cervical clear cell adenocarcinoma in women exposed in utero. The adverse effect is less severe in sons, although links to testicular cancer and urogenital anomalies, such as epididymal cysts, have been reported. 2 3 A recent study also found an increased risk of hypospadias in the sons of women exposed to diethylstilbestrol in utero. 4 Clomifene has a half life of about five days, but its metabolites have been found in blood samples on day 22 of the menstrual cycle and in faeces up to six weeks after administration. 5 The occurrence of hypospadias may be increasing. Little is known about the risk of hypospadias in boys born to women who have used clomifene to induce ovulation. # # # [UNK] and results [UNK] case - control study was done in the [UNK] counties of North [UNK], Aarhus

## **Train**

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

In [22]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [23]:
# 2. Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/scibert_scifact_3class",
    
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="steps",
    save_steps=30,
    
    learning_rate=2e-5,
    warmup_ratio=0.1,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    
    logging_steps=10,
    save_total_limit=1,
    report_to="none",
    
    fp16=torch.cuda.is_available()
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [24]:
# Compute class weights based on training distribution
# Order matches label2id: [CONTRADICT (0), SUPPORT (1), NEI (2)]
class_counts = train_df["label_id"].value_counts().sort_index().values
total_samples = len(train_df)
class_weights = total_samples / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Custom Trainer to apply weighted CrossEntropyLoss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [25]:
# 4. Train
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
30,1.087496,1.039223,0.502793,0.528953,0.506678,0.498514
60,0.891461,0.904381,0.603352,0.585434,0.594091,0.586819
90,0.799009,0.881547,0.642458,0.611094,0.617317,0.610178
120,0.600241,0.901836,0.625698,0.601338,0.605116,0.601577
150,0.420444,0.887380,0.636872,0.634236,0.619480,0.622358
180,0.383535,0.924680,0.659218,0.648455,0.609200,0.616377
210,0.264846,0.930845,0.681564,0.670498,0.669281,0.667547
240,0.160145,1.146662,0.687151,0.640701,0.623417,0.619974
270,0.193038,1.208086,0.692737,0.668786,0.627922,0.625895
300,0.174759,1.108408,0.681564,0.660557,0.645026,0.649918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=300, training_loss=0.5282584031422933, metrics={'train_runtime': 343.0161, 'train_samples_per_second': 16.676, 'train_steps_per_second': 1.05, 'total_flos': 1255051002562560.0, 'train_loss': 0.5282584031422933, 'epoch': 6.666666666666667})

## **Evaluation**

In [26]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.9313083291053772, 'eval_accuracy': 0.6815642458100558, 'eval_precision_macro': 0.6704980842911876, 'eval_recall_macro': 0.6692811692811693, 'eval_f1_macro': 0.6675466636102373, 'eval_runtime': 3.3172, 'eval_samples_per_second': 53.962, 'eval_steps_per_second': 1.809, 'epoch': 6.666666666666667}


In [27]:
predictions = trainer.predict(val_dataset)

logits = predictions.predictions
true_labels = predictions.label_ids

pred_labels = np.argmax(
    logits,
    axis=-1
)

print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=[
            "CONTRADICT",
            "SUPPORT",
            "NEI"
        ],
        digits=4
    )
)

              precision    recall  f1-score   support

  CONTRADICT     0.5000    0.5897    0.5412        39
     SUPPORT     0.6667    0.6757    0.6711        74
         NEI     0.8448    0.7424    0.7903        66

    accuracy                         0.6816       179
   macro avg     0.6705    0.6693    0.6675       179
weighted avg     0.6960    0.6816    0.6868       179



In [28]:
cm = confusion_matrix(
    true_labels,
    pred_labels
)

print(cm)

[[23 12  4]
 [19 50  5]
 [ 4 13 49]]


## **Save**

In [29]:
MODEL_SAVE_PATH = (
    "/kaggle/working/"
    "scibert_scifact_200_classifier"
)

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model saved at:")
print(MODEL_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at:
/kaggle/working/scibert_scifact_200_classifier


In [30]:
preprocessing_config = {
    "document_token_limit": 200,
    "tokenizer": MODEL_NAME,
    "label2id": label2id,
    "id2label": id2label,
    "aggregation_rule": (
        "SUPPORT if all evidence sentences are SUPPORT; "
        "CONTRADICT if all evidence sentences are CONTRADICT; "
        "mixed-label documents skipped"
    )
}

with open(
    os.path.join(
        MODEL_SAVE_PATH,
        "preprocessing_config.json"
    ),
    "w"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

In [31]:
print("done")

done


In [32]:
!zip -r /kaggle/working/scibert_scifact_200_classifier.zip \
    /kaggle/working/scibert_scifact_200_classifier

  adding: kaggle/working/scibert_scifact_200_classifier/ (stored 0%)
  adding: kaggle/working/scibert_scifact_200_classifier/training_args.bin (deflated 53%)
  adding: kaggle/working/scibert_scifact_200_classifier/tokenizer_config.json (deflated 42%)
  adding: kaggle/working/scibert_scifact_200_classifier/config.json (deflated 52%)
  adding: kaggle/working/scibert_scifact_200_classifier/tokenizer.json (deflated 71%)
  adding: kaggle/working/scibert_scifact_200_classifier/preprocessing_config.json (deflated 49%)
  adding: kaggle/working/scibert_scifact_200_classifier/model.safetensors (deflated 7%)


## **Grid Search**